## Data source:

Website: 'https://www.salsify.com/'

## Data preprocessing

#### Step 1 — Create `features_en.json` with aligned title-description records

In [1]:
"""Build flat legacy fields and aligned records from the source catalog."""

import html
import json
from pathlib import Path

LOCALE = "en-GB"


def english_contents(catalog):
    """Yield product-level and variant-override English content blocks."""
    for product in catalog["products"]:
        yield product["data"]["content"].get(LOCALE, {})
        for variant in product.get("variants", []):
            overrides = variant.get("overrides") or {}
            yield overrides.get("content", {}).get(LOCALE, {})


def flatten(catalog, field):
    """Keep the original flat fields for existing consumers."""
    return [text for content in english_contents(catalog) for text in content.get(field) or []]


def build_attribute_records(catalog):
    """Return title records and descriptions that cannot safely be paired."""
    descriptions_by_title = {}
    unpaired_descriptions = []

    for content in english_contents(catalog):
        titles = [html.unescape(text).strip() for text in content.get("feature_titles") or []]
        descriptions = [html.unescape(text).strip() for text in content.get("features") or []]

        for title in titles:
            if title:
                descriptions_by_title.setdefault(title, [])

        # A mismatched block cannot safely associate titles with descriptions.
        if len(titles) != len(descriptions):
            unpaired_descriptions.extend(description for description in descriptions if description)
            continue

        for title, description in zip(titles, descriptions):
            if title and description:
                descriptions_by_title[title].append(description)

    records = [
        {"title": title, "descriptions": descriptions}
        for title, descriptions in descriptions_by_title.items()
    ]
    return records, unpaired_descriptions


source_path = Path("../data_org/catalog.json")
if not source_path.exists():
    raise FileNotFoundError(f"Source catalog not found: {source_path.resolve()}")

with source_path.open(encoding="utf8") as source_file:
    catalog = json.load(source_file)

attribute_records, unpaired_descriptions = build_attribute_records(catalog)
result = {
    "feature_titles": flatten(catalog, "feature_titles"),
    "features": flatten(catalog, "features"),
    "attribute_records": attribute_records,
    "unpaired_descriptions": unpaired_descriptions,
}

output_path = Path("features_en.json")
with output_path.open("w", encoding="utf8") as output_file:
    json.dump(result, output_file, ensure_ascii=False, indent=1)

print(f"feature_titles: {len(result['feature_titles'])} total")
print(f"features: {len(result['features'])} total")
print(f"attribute_records: {len(result['attribute_records'])} unique titles")
print(f"unpaired_descriptions: {len(result['unpaired_descriptions'])} kept for IDF only")

feature_titles: 1078 total
features: 1064 total
attribute_records: 445 unique titles
unpaired_descriptions: 22 kept for IDF only


#### Step 2 — Load the unique attributes and their aligned descriptions

In [2]:
with open("features_en.json", encoding="utf8") as input_file:
    feature_data = json.load(input_file)

attribute_records = feature_data["attribute_records"]
attributes = [record["title"] for record in attribute_records]
descriptions = [record["descriptions"] for record in attribute_records]
print(f"Loaded {len(attributes)} unique attributes")

Loaded 445 unique attributes


## Exploratory Data Analysis